In [1]:
import sqlite3, pandas as pd, numpy as np, matplotlib.pyplot as plt, time
%matplotlib inline
np.random.seed(42)
TITANIC_URL = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'

conn = sqlite3.connect(':memory:')

# products
pd.DataFrame({
    'product_id': range(1, 11),
    'name':     ['Laptop HP','Mysz Logitech','Klawiatura mech.','Monitor 27"',
                 'Sluchawki Sony','Kamera IP','Router Wi-Fi 6','SSD 1TB',
                 'Hub USB-C','Webcam HD'],
    'category': ['Elektronika','Akcesoria','Akcesoria','Elektronika','Audio',
                 'Sprzet','Sprzet','Komponenty','Akcesoria','Sprzet'],
    'price':    [3500, 80, 250, 1200, 350, 600, 280, 320, 90, 150],
    'stock':    [12, 150, 80, 25, 60, 20, 45, 90, 200, 70],
}).to_sql('products', conn, index=False)

# customers + orders
pd.DataFrame({
    'customer_id': [1, 2, 3, 4, 5],
    'name': ['Jan Kowalski','Anna Nowak','Piotr Lis','Maria Zajac','Tomasz Woz'],
    'city': ['Warszawa','Krakow','Gdansk','Wroclaw','Poznan'],
}).to_sql('customers', conn, index=False)

pd.DataFrame({
    'order_id':    [101, 102, 103, 104, 105, 106, 107],
    'customer_id': [1,   2,   1,   3,   1,   2,   3],
    'order_date':  ['2024-01-10','2024-01-11','2024-02-05',
                   '2024-02-10','2024-03-01','2024-03-15','2024-03-20'],
    'amount':      [250.0, 120.5, 340.0, 80.0, 190.0, 450.0, 220.0],
}).to_sql('orders', conn, index=False)

# order_items
pd.DataFrame({
    'item_id':  [1, 2, 3, 4, 5, 6, 7],
    'order_id': [101,101,102,102,103,104,107],
    'product':  ['Laptop','Mysz','Klawiatura','Monitor','Sluchawki','Kamera','SSD'],
    'quantity': [1, 2, 1, 1, 2, 1, 1],
    'price':    [3500,80,250,1200,350,600,320],
}).to_sql('order_items', conn, index=False)

# employees
pd.DataFrame({
    'employee_id': [1,2,3,4,5,6,7,8],
    'name':  ['Anna Dyrektor','Piotr CFO','Marek CTO','Ewa Dev Lead',
              'Jan Analityk','Sara Developer','Adam Ksiegowy','Lena QA'],
    'title': ['CEO','CFO','CTO','Dev Lead','Analityk','Developer','Ksiegowy','QA'],
    'manager_id': [None,1,1,3,2,4,2,4],
}).to_sql('employees', conn, index=False)

# sales (1000 wierszy z datami 2024)
dates = pd.date_range('2024-01-01','2024-12-31').astype(str)
pd.DataFrame({
    'sale_id':   range(1, 1001),
    'sale_date': np.random.choice(dates, 1000),
    'product':   np.random.choice(['Laptop','Mysz','Monitor','Klawiatura','Sluchawki'],1000),
    'category':  np.random.choice(['Elektronika','Akcesoria','Audio'],1000),
    'revenue':   np.random.randint(50, 3500, 1000),
}).to_sql('sales', conn, index=False)

# customers_big (100) + orders_big (500)
pd.DataFrame({
    'customer_id':       range(1, 101),
    'name':              [f'Klient_{i}' for i in range(1,101)],
    'registration_date': pd.date_range('2024-01-01',periods=100,freq='3D').astype(str),
    'country':           np.random.choice(['Polska','Niemcy','Francja','UK','Czechy'],100),
}).to_sql('customers_big', conn, index=False)

pd.DataFrame({
    'order_id':    range(1, 501),
    'customer_id': np.random.randint(1, 101, 500),
    'order_date':  pd.date_range('2024-01-01',periods=500,freq='12h').astype(str),
    'amount':      np.random.uniform(50, 2000, 500).round(2),
}).to_sql('orders_big', conn, index=False)

# passengers (Titanic)
pd.read_csv(TITANIC_URL).to_sql('passengers', conn, index=False)

print('Tabele gotowe:')
for t in pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'",conn)['name']:
    n = pd.read_sql_query(f'SELECT COUNT(*) AS n FROM {t}',conn).iloc[0,0]
    print(f'  {t:<20} {n:>5} wierszy')

Tabele gotowe:
  products                10 wierszy
  customers                5 wierszy
  orders                   7 wierszy
  order_items              7 wierszy
  employees                8 wierszy
  sales                 1000 wierszy
  customers_big          100 wierszy
  orders_big             500 wierszy
  passengers             891 wierszy


### ✏️ Zadanie 5 – LEFT JOIN

Używając tabel z zadania 4:

**Wymagania:**
- Wykonaj LEFT JOIN pokazujący WSZYSTKICH klientów (nawet bez zamówień)
- Użyj COALESCE aby zamienić NULL na 0 w kwocie zamówienia
- Znajdź klientów którzy NIE złożyli zamówienia (`WHERE order_id IS NULL`)

*(proste)*

In [7]:
customer_orders = pd.read_sql_query("""
SELECT c.customer_id, c.name, COALESCE(o.amount, 0) AS order_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
""", conn)

customers_no_orders = pd.read_sql_query("""
SELECT c.customer_id, c.name, COALESCE(o.amount, 0) AS order_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
WHERE o.order_id IS NULL
""", conn)

customer_orders, customers_no_orders

(   customer_id          name  order_amount
 0            1  Jan Kowalski         190.0
 1            1  Jan Kowalski         250.0
 2            1  Jan Kowalski         340.0
 3            2    Anna Nowak         120.5
 4            2    Anna Nowak         450.0
 5            3     Piotr Lis          80.0
 6            3     Piotr Lis         220.0
 7            4   Maria Zajac           0.0
 8            5    Tomasz Woz           0.0,
    customer_id         name  order_amount
 0            4  Maria Zajac             0
 1            5   Tomasz Woz             0)

### ✏️ Zadanie 6 – Podstawowy GROUP BY

Używając tabeli `orders`:

**Wymagania:**
- Policz liczbę zamówień dla każdego klienta
- Oblicz łączną kwotę zamówień dla każdego klienta
- Oblicz średnią wartość zamówienia dla każdego klienta

*(proste)*

In [10]:
client_order_summary = pd.read_sql_query("""
SELECT c.customer_id, c.name,
       COUNT(o.order_id) AS order_count,
       SUM(o.amount) AS total_amount,
       AVG(o.amount) AS avg_amount
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name
""", conn)

client_order_summary

,customer_id,name,order_count,total_amount,avg_amount
0,1,Jan Kowalski,3,780.0,260.00
1,2,Anna Nowak,2,570.5,285.25
2,3,Piotr Lis,2,300.0,150.00
3,4,Maria Zajac,0,NaN,NaN
4,5,Tomasz Woz,0,NaN,NaN


### ✏️ Zadanie 7 – COUNT i SUM

**Wymagania:**
- Policz łączną liczbę produktów w tabeli `products`
- Oblicz łączną wartość zapasów (`stock * price`) dla wszystkich produktów
- Oblicz średnią cenę produktów w każdej kategorii

*(proste)*

In [17]:
all_product_summary = pd.read_sql_query("""
SELECT
    COUNT(*) AS total_products,
    SUM(stock * price) AS total_stock_value,
    AVG(price) AS avg_price
FROM products
""", conn)

avg_product_cat_price =pd.read_sql_query("""
SELECT
    category,
    ROUND(AVG(price), 2) AS avg_price
FROM products
GROUP BY category
""", conn)

all_product_summary, avg_product_cat_price

(   total_products  total_stock_value  avg_price
 0              10             206900      682.0,
       category  avg_price
 0    Akcesoria     140.00
 1        Audio     350.00
 2  Elektronika    2350.00
 3   Komponenty     320.00
 4       Sprzet     343.33)

### ✏️ Zadanie 8 – Tworzenie indeksu

**Wymagania:**
- Utwórz tabelę z 10 000 rekordami (użyj pętli lub NumPy)
- Zmierz czas zapytania SELECT z filtrem WHERE **BEZ** indeksu
- Utwórz indeks na kolumnie filtrowanej
- Zmierz czas zapytania **Z** indeksem
- Porównaj wyniki

*(proste)*

> Osobne połączenie `conn_bench` — nie zaśmieca głównej bazy.

In [18]:
conn_bench = sqlite3.connect(':memory:')
table = pd.DataFrame({
    'id': range(1, 10001),
    'value': np.random.randint(1, 10000, 10000)
})

table.to_sql('bench_table', conn_bench, index=False)

10000

In [20]:
# No index

start = time.perf_counter()

pd.read_sql(
    "SELECT * FROM bench_table WHERE value = 5000",
    conn_bench
)

end = time.perf_counter()

print(f"Bez indeksu: {end - start:.8f} s")

Bez indeksu: 0.00105780 s


In [21]:
# With index
cursor = conn_bench.cursor()

cursor.execute("""
CREATE INDEX idx_value
ON bench_table(value)
""")

conn_bench.commit()


start = time.perf_counter()

pd.read_sql(
    "SELECT * FROM bench_table WHERE value = 5000",
    conn_bench
)

end = time.perf_counter()

print(f"Z indeksem: {end - start:.8f} s")

Z indeksem: 0.00068000 s


### 🧠 Zadanie 18 – Self-JOIN

**Wymagania:**
- Utwórz tabelę `employees` z kolumnami: `employee_id`, `name`, `manager_id`
- Użyj self-JOIN aby wyświetlić pracownika i jego managera:
  `SELECT e1.name AS employee, e2.name AS manager FROM employees e1 LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id`
- Znajdź wszystkich pracowników bez managera (TOP management)
- Policz liczbę podwładnych dla każdego managera

**Dataset:** Własny (hierarchia)

*(challenge)*

In [22]:
employee_manager = pd.read_sql_query("""
    SELECT e1.name AS employee, e2.name AS manager
    FROM employees e1
    LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
""", conn)

employee_no_manager = pd.read_sql_query("""
    SELECT e1.name AS employee
    FROM employees e1
    LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
    WHERE e2.employee_id IS NULL
""", conn)

manager_employee_count = pd.read_sql_query("""
    SELECT e2.name AS manager, COUNT(e1.employee_id) AS num_employees
    FROM employees e1
    LEFT JOIN employees e2 ON e1.manager_id = e2.employee_id
    GROUP BY e2.employee_id, e2.name
""", conn)

employee_manager, employee_no_manager, manager_employee_count

(         employee        manager
 0   Anna Dyrektor            NaN
 1       Piotr CFO  Anna Dyrektor
 2       Marek CTO  Anna Dyrektor
 3    Ewa Dev Lead      Marek CTO
 4    Jan Analityk      Piotr CFO
 5  Sara Developer   Ewa Dev Lead
 6   Adam Ksiegowy      Piotr CFO
 7         Lena QA   Ewa Dev Lead,
         employee
 0  Anna Dyrektor,
          manager  num_employees
 0            NaN              1
 1  Anna Dyrektor              2
 2      Piotr CFO              2
 3      Marek CTO              1
 4   Ewa Dev Lead              2)